# 02 — Feature Engineering: NASA IMS Bearing Dataset

**Project:** robotic-bearing-pdm  
**Goal:** Build the final feature matrix used to train the LSTM Autoencoder.

Pipeline in this notebook:
```
Raw signals (n_files, 20480, 4)
    → Time-domain features    (RMS, kurtosis, crest factor, peak-to-peak, skewness, shape factor)
    → Frequency-domain features (3 band energies, spectral entropy, dominant frequency)
    → Rolling statistics      (rolling mean + std over 2h and 6h windows)
    → Z-score normalisation   (fit on healthy window only)
    → Saved to data/processed/features_final.csv
```

## 0 · Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.data.loader import load_snapshots, train_test_split_temporal, create_windows
from src.data.features import (
    extract_all_features,
    add_rolling_features,
    fit_scaler,
    apply_scaler,
)

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
PROC_DIR.mkdir(exist_ok=True)

TRAIN_FRAC = 0.20   # first 20% = healthy reference for training
SEQ_LEN    = 50     # LSTM input window size (snapshots)
STEP       = 1      # sliding window stride

print(f"Root: {ROOT}")

## 1 · Load Raw Signals

In [ ]:
signals, timestamps, bearing_names = load_snapshots(DATA_DIR, dataset=2)
print(f"Signals shape : {signals.shape}")
print(f"Bearings      : {bearing_names}")
print(f"Date range    : {timestamps[0]}  →  {timestamps[-1]}")

## 2 · Extract Time + Frequency Features

11 features per bearing × 4 bearings = **44 raw features** per snapshot.

In [ ]:
df_raw = extract_all_features(signals, timestamps, bearing_names)

print(f"Raw feature matrix : {df_raw.shape}")
print(f"Columns            : {list(df_raw.columns)}")
df_raw.head(3)

## 3 · Add Rolling Features

Rolling mean and std over **12-snapshot (~2 h)** and **36-snapshot (~6 h)** windows  
applied to RMS, kurtosis, and crest factor — the three most diagnostic features.

This gives the LSTM temporal context beyond the raw instantaneous value.

In [ ]:
df_features = add_rolling_features(
    df_raw,
    base_features=["rms", "kurtosis", "crest_factor"],
    windows=[12, 36],
)

print(f"Feature matrix with rolling stats : {df_features.shape}")
print(f"Total features per timestep       : {df_features.shape[1]}")

## 4 · Feature Distribution Analysis

Compare feature distributions between healthy and degraded periods.

In [ ]:
n_files   = len(df_features)
split_idx = int(n_files * TRAIN_FRAC)

df_healthy = df_features.iloc[:split_idx]
df_test    = df_features.iloc[split_idx:]

# Plot distributions for the 6 raw time-domain features of bearing_1
b1_raw_feats = ["bearing_1_rms", "bearing_1_kurtosis", "bearing_1_crest_factor",
                "bearing_1_peak_to_peak", "bearing_1_skewness", "bearing_1_shape_factor"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for ax, feat in zip(axes, b1_raw_feats):
    ax.hist(df_healthy[feat], bins=30, alpha=0.6, label="Healthy",  color="steelblue", density=True)
    ax.hist(df_test[feat],    bins=30, alpha=0.6, label="Test+Fail", color="tomato",    density=True)
    ax.set_title(feat.replace("bearing_1_", ""), fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle("Bearing 1 — Feature Distributions: Healthy vs Test Window", fontsize=11)
plt.tight_layout()
plt.savefig(PROC_DIR / "07_feature_distributions.png", bbox_inches="tight")
plt.show()

## 5 · Feature Importance (Separability)

Measure how well each feature separates healthy from degraded periods  
using the **Kolmogorov-Smirnov statistic** — higher = more discriminative.

In [ ]:
from scipy.stats import ks_2samp

ks_scores = {}
for col in df_raw.columns:   # use raw features only for interpretability
    stat, _ = ks_2samp(df_healthy[col].dropna(), df_test[col].dropna())
    ks_scores[col] = stat

ks_df = pd.Series(ks_scores).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
ks_df.head(20).plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Top 20 Features by KS Separability (healthy vs degraded)", fontsize=11)
ax.set_ylabel("KS Statistic")
ax.set_xlabel("Feature")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(PROC_DIR / "08_feature_importance_ks.png", bbox_inches="tight")
plt.show()

print("\nTop 10 most discriminative features:")
print(ks_df.head(10).to_string())

## 6 · Normalisation

Z-score normalise using **only the healthy (training) window** statistics.  
This is critical — leaking test-window stats would bias the anomaly threshold.

In [ ]:
X = df_features.values.astype(np.float32)
X_train_raw, X_test_raw, ts_train, ts_test = train_test_split_temporal(
    X, list(df_features.index), train_frac=TRAIN_FRAC
)

mu, sigma = fit_scaler(X_train_raw)
X_train   = apply_scaler(X_train_raw, mu, sigma)
X_test    = apply_scaler(X_test_raw,  mu, sigma)

print(f"X_train shape : {X_train.shape}  (healthy — LSTM training data)")
print(f"X_test  shape : {X_test.shape}   (full run including failure)")
print(f"\nTrain stats after scaling — mean: {X_train.mean():.4f}, std: {X_train.std():.4f}")

## 7 · Create LSTM Windows

Sliding window of `seq_len=50` snapshots, stride=1.  
Each window = 50 × 10 min = **~8.3 hours** of history seen by the LSTM.

In [ ]:
windows_train = create_windows(X_train, seq_len=SEQ_LEN, step=STEP)
windows_test  = create_windows(X_test,  seq_len=SEQ_LEN, step=STEP)

print(f"Training windows : {windows_train.shape}  (n_windows, seq_len, n_features)")
print(f"Test windows     : {windows_test.shape}")
print(f"\nLSTM input shape : {windows_train.shape[1:]}  → [batch, {SEQ_LEN}, {windows_train.shape[2]}]")

## 8 · Rolling Feature Trends Visualisation

The rolling features smooth out noise and capture the gradual degradation ramp.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

for col, bname in enumerate(bearing_names):
    ax = axes[col // 2, col % 2]
    raw_col   = f"{bname}_rms"
    roll12    = f"{bname}_rms_roll12_mean"
    roll36    = f"{bname}_rms_roll36_mean"

    ax.plot(df_features.index, df_features[raw_col],  lw=0.5, alpha=0.4, label="Raw RMS", color=f"C{col}")
    ax.plot(df_features.index, df_features[roll12],   lw=1.2, label="Roll-12 (~2h)", color=f"C{col}")
    ax.plot(df_features.index, df_features[roll36],   lw=1.5, ls="--", label="Roll-36 (~6h)", color=f"C{col}")
    ax.axvspan(df_features.index[0], df_features.index[split_idx],
               alpha=0.07, color="green", label="Train window")
    ax.set_title(f"{bname} — RMS with rolling averages", fontsize=9)
    ax.legend(fontsize=7)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.suptitle("RMS + Rolling Averages — All Bearings", fontsize=11)
plt.tight_layout()
plt.savefig(PROC_DIR / "09_rolling_rms_all_bearings.png", bbox_inches="tight")
plt.show()

## 9 · Save Artefacts

In [ ]:
import json

# Full feature matrix (for inspection / re-use)
df_features.to_csv(PROC_DIR / "features_final.csv")

# Normalised numpy arrays (ready for model training)
np.save(PROC_DIR / "X_train.npy",        X_train)
np.save(PROC_DIR / "X_test.npy",         X_test)
np.save(PROC_DIR / "windows_train.npy",  windows_train)
np.save(PROC_DIR / "windows_test.npy",   windows_test)

# Scaler parameters (needed at inference time)
scaler = {"mu": mu.tolist(), "sigma": sigma.tolist()}
with open(PROC_DIR / "scaler.json", "w") as f:
    json.dump(scaler, f)

# Feature column names (needed to reconstruct DataFrame at inference)
with open(PROC_DIR / "feature_names.json", "w") as f:
    json.dump(list(df_features.columns), f)

print("Saved artefacts:")
for p in sorted(PROC_DIR.iterdir()):
    print(f"  {p.name:40s}  {p.stat().st_size / 1e3:.1f} KB")

## 10 · Summary

| Artefact | Shape / Size | Used by |
|---|---|---|
| `features_final.csv` | `(n_files, n_features)` | Inspection, EDA |
| `X_train.npy` | `(n_train, n_features)` float32 | Model training |
| `X_test.npy` | `(n_test, n_features)` float32 | Evaluation |
| `windows_train.npy` | `(n_windows, 50, n_features)` | LSTM DataLoader |
| `windows_test.npy` | `(n_windows, 50, n_features)` | LSTM evaluation |
| `scaler.json` | `{mu, sigma}` vectors | FastAPI inference |
| `feature_names.json` | list of strings | FastAPI inference |

**Next:** `03_model_training.ipynb` — train the LSTM Autoencoder on `windows_train.npy`.